# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")


## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# List all available record sets and their fields using @id

print('Available record sets:')
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '(no name)')}")
    print("  Fields:")
    for field in record_set.get('field', []):
        if isinstance(field, dict):
            print(f"    - Field @id: {field.get('@id')}, name: {field.get('name')}, dataType: {field.get('dataType')}")
        else:
            print(f"    - Field Reference: {field}")
    print('')

# Capture all record_set IDs for next step
record_set_ids = [rs['@id'] for rs in dataset.record_sets]


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references are made using the record set and field `@id`s.


In [ ]:
# Extract data from each record set using their @id
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  - Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  - No records found.")

# For demonstration, select first available record set for further analysis
if len(dataframes) == 0:
    raise ValueError("No record sets contained records for analysis.")
primary_record_set_id = list(dataframes.keys())[0]
print(f"\nUsing RecordSet @id for EDA: {primary_record_set_id}\n")
print(f"Record columns: {dataframes[primary_record_set_id].columns.tolist()}")
dataframes[primary_record_set_id].head()


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes to prepare for further analysis.


In [ ]:
df = dataframes[primary_record_set_id]

# Find available numeric columns (float or int)
numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
print(f"Numeric columns available: {numeric_columns}")

if not numeric_columns:
    print("No numeric columns found for EDA.")
else:
    numeric_field = numeric_columns[0]  # Use the first numeric field found
    print(f"Using field '@id': {numeric_field} for filtering and normalization.")

    # Example: filter where numeric_field > its median
    threshold = df[numeric_field].median()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records where {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field if available
    candidate_group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype=='object']
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"\nGrouping by: '{group_field}'")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable group field found for aggregation.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_columns:
    print("No numeric columns to visualize.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatterplot if another numeric field exists
    if len(numeric_columns) > 1:
        y_field = numeric_columns[1]
        plt.figure(figsize=(7,5))
        sns.scatterplot(x=df[numeric_field], y=df[y_field])
        plt.xlabel(numeric_field)
        plt.ylabel(y_field)
        plt.title(f"Scatterplot of {numeric_field} vs {y_field}")
        plt.show()
    else:
        print('Only one numeric field for histogram.')


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


**Summary:**

- Loaded metadata and data records from the FAIR^2 dataset via Croissant schema.
- Inspected available record sets and fields by their `@id`.
- Extracted data and performed basic filtering and normalization on a key numeric field.
- Visualized statistical distributions for deeper exploratory analysis.

This notebook provides a reproducible template for programmatic exploration and processing using `mlcroissant`, referencing all entities by `@id` as recommended.
